# SITS Annotator - Coleta de Amostras

Este notebook inicia o software de anotacao para coletar amostras de series temporais.

**Workflow:**
1. Configurar caminho do projeto (YAML)
2. Verificar dados (stack, mascara, classes)
3. Executar o anotador
4. Visualizar estatisticas
5. Exportar para NPZ (opcional)

**Saidas:**
- `cultivos.json`: Anotacoes brutas
- `cultivos.npz`: Arrays numpy (X, y) prontos para treinamento

## 1. Configuracao

Defina o caminho para o arquivo de configuracao do projeto.

In [ ]:
from pathlib import Path

# === CONFIGURE AQUI ===
CONFIG_FILE = "../config/seu_projeto.yaml"
# ======================

config_path = Path(CONFIG_FILE).resolve()

if config_path.exists():
    print(f"Configuracao encontrada: {config_path}")
else:
    print(f"ERRO: Configuracao NAO encontrada: {config_path}")
    print("Verifique o caminho CONFIG_FILE acima.")

## 2. Verificar Dados

Carrega a configuracao e verifica se os arquivos existem.

In [ ]:
import yaml

# Diretorio raiz do projeto (para resolver caminhos relativos)
project_root = config_path.parent.parent

# Carregar configuracao
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(f"Projeto: {config['project_name']}")
print(f"Session folder: {config['session_folder']}")
print()

# Verificar stack
stack_path = project_root / config['stack']['path']
if stack_path.exists():
    print(f"Stack: {stack_path}")
else:
    print(f"ERRO - Stack NAO encontrado: {stack_path}")

# Verificar mascara (se existir)
if 'auxiliary_mask' in config:
    mask_path = project_root / config['auxiliary_mask']['path']
    if mask_path.exists():
        print(f"Mascara: {mask_path}")
    else:
        print(f"ERRO - Mascara NAO encontrada: {mask_path}")

# Pasta de anotacoes
annotation_folder = project_root / config['session_folder'] / "annotation"
output_filename = config.get('output', {}).get('annotations_filename', 'annotations.json')
print(f"\nPasta de saida: {annotation_folder}")
print(f"Arquivo de anotacoes: {output_filename}")

In [ ]:
# Classes de anotacao
print("Classes de anotacao:")
print("-" * 50)
for cls in config['annotation_classes']:
    print(f"  [{cls['shortcut']}] {cls['name']:15s} - {cls['description']}")

# Datas da serie temporal
print()
print(f"Serie temporal: {config['stack']['n_times']} datas")
print("-" * 50)
for i, date in enumerate(config['stack']['dates'], 1):
    print(f"  {i:2d}. {date}")

## 3. Iniciar Anotador

Execute a celula abaixo para abrir o SITS Annotator.

**Atalhos:**
- `1`, `2`, `3`... : Classificar na classe correspondente
- `D`: Nao sei (dont_know)
- `X`: Pular (skip)
- `Setas`: Navegar entre amostras
- `S`: Mostrar/ocultar similaridade
- `V`: Alternar visualizacao (NDVI/EVI/etc)
- `M`: Alternar mascara

In [ ]:
import subprocess
import sys
import os

print(f"Diretorio do projeto: {project_root}")
print(f"Config: {config_path}")
print()
print("Iniciando SITS Annotator...")
print("(Feche o software para continuar)")
print()

# Executar a partir da raiz do projeto
original_cwd = os.getcwd()
try:
    os.chdir(project_root)
    result = subprocess.run([sys.executable, "main.py", str(config_path)])
finally:
    os.chdir(original_cwd)

print()
print("Anotador encerrado.")

## 4. Estatisticas das Anotacoes

In [ ]:
import json

output_file = annotation_folder / output_filename

if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    metadata = data.get("metadata", {})
    statistics = data.get("statistics", {})
    samples = data.get("samples", [])
    
    print(f"Arquivo: {output_file.name}")
    print(f"Ultima modificacao: {metadata.get('last_modified', 'N/A')}")
    print(f"Total de amostras: {len(samples)}")
    print()
    print("Distribuicao por classe:")
    print("-" * 35)
    for class_name, count in statistics.items():
        pct = (count / len(samples) * 100) if samples else 0
        print(f"  {class_name:20s}: {count:5d} ({pct:5.1f}%)")
    print("-" * 35)
    print(f"  {'TOTAL':20s}: {len(samples):5d}")
else:
    print(f"Nenhuma anotacao encontrada em: {output_file}")
    print("Execute o anotador primeiro (celula anterior).")

In [ ]:
# Visualizacao grafica
import matplotlib.pyplot as plt

if output_file.exists() and statistics:
    # Filtrar classes com amostras
    classes = [k for k, v in statistics.items() if v > 0]
    counts = [statistics[k] for k in classes]
    
    if classes:
        # Cores do config
        color_map = {c['name']: c['color'] for c in config['annotation_classes']}
        colors = [color_map.get(c, '#666666') for c in classes]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Grafico de barras
        bars = ax1.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
        ax1.set_xlabel("Classe")
        ax1.set_ylabel("Quantidade")
        ax1.set_title("Distribuicao das Anotacoes")
        ax1.tick_params(axis='x', rotation=45)
        for bar, count in zip(bars, counts):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(count), ha='center', va='bottom', fontsize=10)
        
        # Grafico de pizza
        ax2.pie(counts, labels=classes, colors=colors, autopct='%1.1f%%',
                startangle=90, explode=[0.02]*len(classes))
        ax2.set_title("Proporcao por Classe")
        
        plt.tight_layout()
        plt.show()
    else:
        print("Nenhuma classe com amostras para visualizar.")
else:
    print("Sem dados para visualizar.")

## 5. Arquivos Auxiliares

Verifica amostras marcadas como "nao sei" ou "pular".

In [ ]:
# Arquivos auxiliares
dont_know_filename = config.get('output', {}).get('dont_know_filename', 'dont_know.json')
skipped_filename = config.get('output', {}).get('skipped_filename', 'skipped.json')

print("Arquivos auxiliares:")
print("-" * 45)

for filename in [dont_know_filename, skipped_filename]:
    filepath = annotation_folder / filename
    if filepath.exists():
        with open(filepath, "r", encoding="utf-8") as f:
            aux_data = json.load(f)
        count = len(aux_data.get("samples", []))
        print(f"  {filename:25s}: {count:5d} amostras")
    else:
        print(f"  {filename:25s}: nao existe")

print()
print(f"Pasta: {annotation_folder}")

## 6. Exportar para NPZ (Opcional)

Converte as anotacoes JSON para formato NPZ, pronto para treinamento.

In [ ]:
import numpy as np

# Parametros do stack
NUM_CHANNELS = len(config['stack']['bands'])
NUM_TIMESTEPS = config['stack']['n_times']
BAND_NAMES = [b['name'].lower() for b in config['stack']['bands']]

print(f"Parametros do stack:")
print(f"  Canais: {NUM_CHANNELS} ({BAND_NAMES})")
print(f"  Timesteps: {NUM_TIMESTEPS}")

In [ ]:
def convert_json_to_npz(json_path, num_channels, num_timesteps, band_names):
    """Converte anotacoes JSON para arrays numpy."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    samples = data.get("samples", [])
    if not samples:
        raise ValueError("Nenhuma amostra encontrada no JSON")
    
    # Criar mapeamento de classes
    class_names = sorted(set(s["class_name"] for s in samples))
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    # Arrays de saida
    X = np.zeros((len(samples), num_channels, num_timesteps), dtype=np.float32)
    y = np.zeros(len(samples), dtype=np.int64)
    
    for i, sample in enumerate(samples):
        # Label
        y[i] = class_mapping[sample["class_name"]]
        
        # Time series
        ts_data = sample.get("time_series", {})
        for c, band in enumerate(band_names):
            if band in ts_data:
                values = ts_data[band][:num_timesteps]
                X[i, c, :len(values)] = values
    
    return X, y, class_mapping

# Converter
if output_file.exists():
    X, y, class_mapping = convert_json_to_npz(
        output_file, NUM_CHANNELS, NUM_TIMESTEPS, BAND_NAMES
    )
    
    print(f"Conversao concluida!")
    print(f"  X shape: {X.shape}  (N, C, T)")
    print(f"  y shape: {y.shape}  (N,)")
    print(f"  X range: [{X.min():.2f}, {X.max():.2f}]")
    print()
    print("Classes:")
    for name, idx in sorted(class_mapping.items(), key=lambda x: x[1]):
        count = np.sum(y == idx)
        print(f"  {idx}: {name:20s} - {count:5d} amostras")
else:
    print("Arquivo de anotacoes nao encontrado.")

In [ ]:
# Salvar NPZ
from datetime import datetime

if output_file.exists():
    # Nome base do arquivo de saida
    base_name = output_file.stem  # ex: "cultivos"
    
    # Salvar NPZ
    npz_path = annotation_folder / f"{base_name}.npz"
    np.savez_compressed(npz_path, X=X, y=y)
    
    # Salvar metadados
    metadata_path = annotation_folder / f"{base_name}_metadata.json"
    idx_to_name = {v: k for k, v in class_mapping.items()}
    
    export_metadata = {
        "source_file": output_file.name,
        "n_samples": int(X.shape[0]),
        "n_channels": int(X.shape[1]),
        "n_timesteps": int(X.shape[2]),
        "band_names": BAND_NAMES,
        "class_mapping": class_mapping,
        "idx_to_name": {str(k): v for k, v in idx_to_name.items()},
        "statistics": {idx_to_name[idx]: int(np.sum(y == idx)) for idx in range(len(class_mapping))},
        "created": datetime.now().isoformat(),
    }
    
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(export_metadata, f, indent=2, ensure_ascii=False)
    
    # Mostrar resultado
    npz_size = npz_path.stat().st_size / 1024
    
    print("Arquivos salvos:")
    print(f"  {npz_path.name}: {npz_size:.1f} KB")
    print(f"  {metadata_path.name}")
    print()
    print(f"Caminho: {annotation_folder}")

In [ ]:
# Verificar integridade
if output_file.exists() and npz_path.exists():
    data_check = np.load(npz_path)
    assert np.allclose(X, data_check["X"]), "X nao corresponde!"
    assert np.array_equal(y, data_check["y"]), "y nao corresponde!"
    print("Verificacao de integridade: OK")

## Proximos Passos

Arquivos gerados:
- `*.json`: Anotacoes brutas (formato do software)
- `*.npz`: Arrays numpy (X, y) prontos para treinamento
- `*_metadata.json`: Mapeamento de classes e estatisticas

Proximos notebooks:
- **02_treinamento.ipynb**: Treinar modelo de classificacao
- **03_classificacao.ipynb**: Aplicar modelo na imagem completa